# 01 — Enrich Artists from rostr.cc

The actual job. Steps:

1. Read the artists tab.
2. For each row missing **any** of `Agency / Agent Name / Management / Management Contact`, look up the artist on rostr.cc.
3. Fill only the blank cells; preserve manual edits.
4. Stamp `Last Updated` (UTC, minute precision).
5. *(Optional)* If `PERSIST_TO_UC=true`, also land raw API responses in a UC Volume + append an audit row to a Delta history table.

This is the notebook that runs nightly under the Databricks job.

In [ ]:
import os, json, datetime as dt
from dotenv import load_dotenv
load_dotenv()

from rostr_client  import RostrClient, RostrAPIError, slugify
from sheets_client import SheetsClient, CellUpdate, COL_AGENCY, COL_AGENT, COL_MGMT, COL_MGMT_CONTACT, COL_UPDATED, now_iso_minute

PERSIST_TO_UC = os.getenv('PERSIST_TO_UC', 'false').lower() == 'true'
print(f'PERSIST_TO_UC = {PERSIST_TO_UC}')

## 1. Connect to both systems

In [ ]:
rostr = RostrClient(
    username=os.environ['ROSTR_USERNAME'],
    password=os.environ['ROSTR_PASSWORD'],
    cookie_path=os.getenv('ROSTR_COOKIE_PATH') or None,
)
rostr.authenticate()

sheets = SheetsClient(
    sheet_id=os.environ['GOOGLE_SHEET_ID'],
    tab=os.getenv('GOOGLE_SHEET_TAB', 'Sheet1'),
    auth_mode=os.getenv('GOOGLE_AUTH_MODE', 'adc'),
    service_account_path=os.getenv('GOOGLE_APPLICATION_CREDENTIALS') or None,
    quota_project=os.getenv('GOOGLE_QUOTA_PROJECT'),
)
sheets.ensure_canonical_header()
rows = sheets.read_artists()
needs = [r for r in rows if r.needs_enrichment()]
print(f'  total rows   : {len(rows)}')
print(f'  to enrich    : {len(needs)}')

## 2. Look up each artist on rostr.cc

We comma-join multiple agents/managers per role, and use the first non-empty
email/phone we find as the contact column.

In [ ]:
def flatten(team) -> dict[str, str]:
    """Compress a list of TeamContact objects into the four sheet columns."""
    agency_rows = team.by_role('AGENCY')
    mgmt_rows   = team.by_role('MANAGEMENT')

    def _join(values, sep=' / '):
        seen, out = set(), []
        for v in values:
            if v and v not in seen:
                seen.add(v); out.append(v)
        return sep.join(out)

    def _first(values):
        for v in values:
            if v:
                return v
        return ''

    return {
        'agency':              _join({c.company for c in agency_rows}),
        'agent_name':          _join([c.person_name for c in agency_rows if c.person_name]),
        'management':          _join({c.company for c in mgmt_rows}),
        'management_contact':  _first([c.person_email or c.person_phone or c.person_name for c in mgmt_rows]),
    }


results: list[tuple] = []  # (artist_row, flattened_or_None, error_or_None)
for row in needs:
    try:
        team = rostr.lookup_artist_team(row.artist, handle_override=row.handle_override)
        flat = flatten(team)
        results.append((row, flat, team, None))
        print(f'  ✓ {row.artist:<30s} -> agency={flat["agency"]!r:<28s} mgmt={flat["management"]!r}')
    except RostrAPIError as e:
        results.append((row, None, None, str(e)))
        print(f'  ✗ {row.artist:<30s} -> {e}')

print(f'\nResolved {sum(1 for r in results if r[1])}/{len(results)} artists.')

## 3. Build per-cell updates

We only write the cells that are currently blank — manual edits in the
sheet are preserved.

In [ ]:
stamp = now_iso_minute()
updates: list[CellUpdate] = []

for row, flat, team, err in results:
    if flat is None:
        continue
    blanks = set(row.blank_columns())
    if COL_AGENCY        in blanks and flat['agency']:             updates.append(CellUpdate(row.row_index, COL_AGENCY,        flat['agency']))
    if COL_AGENT         in blanks and flat['agent_name']:         updates.append(CellUpdate(row.row_index, COL_AGENT,         flat['agent_name']))
    if COL_MGMT          in blanks and flat['management']:         updates.append(CellUpdate(row.row_index, COL_MGMT,          flat['management']))
    if COL_MGMT_CONTACT  in blanks and flat['management_contact']: updates.append(CellUpdate(row.row_index, COL_MGMT_CONTACT,  flat['management_contact']))
    # Always touch Last Updated when we wrote anything.
    if any(u.row_index == row.row_index for u in updates):
        updates.append(CellUpdate(row.row_index, COL_UPDATED, stamp))

print(f'Will write {len(updates)} cells.')
for u in updates[:20]:
    print(f'  {u.to_a1(sheets.tab):<14s} = {u.value!r}')

## 4. Apply updates

In [ ]:
n_written = sheets.apply_updates(updates)
print(f'Wrote {n_written} cells to {sheets.sheet_id}#{sheets.tab}')

## 5. (Optional) Persist to Unity Catalog

Off by default — flip `PERSIST_TO_UC=true` in `.env` if the customer wants
an audit trail / history table.

We land:
* the raw entity JSON for each artist in `/Volumes/{cat}/{schema}/raw_data/{run_ts}/{slug}.json`
* one row per `(run_ts, artist, role)` in a Delta `team_enrichment_history` table.

In [ ]:
if PERSIST_TO_UC:
    try:
        spark
    except NameError:
        from databricks.connect import DatabricksSession
        spark = DatabricksSession.builder.serverless().getOrCreate()
    from databricks.sdk import WorkspaceClient
    w = WorkspaceClient()

    UC_CATALOG = os.getenv('UC_CATALOG', 'alexander_booth')
    UC_SCHEMA  = os.getenv('UC_SCHEMA',  'rostr_artist_enrichment')
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.{UC_SCHEMA}')
    spark.sql(f'CREATE VOLUME IF NOT EXISTS {UC_CATALOG}.{UC_SCHEMA}.raw_data')
    run_ts = dt.datetime.now(dt.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    base   = f'/Volumes/{UC_CATALOG}/{UC_SCHEMA}/raw_data/{run_ts}'

    history_rows: list[dict] = []
    for row, flat, team, err in results:
        slug = slugify(row.artist)
        if team is not None:
            payload = json.dumps({'artist': team.raw_artist, 'teams': team.raw_teams}).encode()
            w.files.upload(f'{base}/{slug}.json', payload, overwrite=True)
        for c in (team.contacts if team else []):
            history_rows.append({
                'run_ts'        : run_ts,
                'artist'        : row.artist,
                'rostr_handle'  : team.handle if team else slug,
                'rostr_id'      : team.rostr_id if team else None,
                'role'          : c.role,
                'company'       : c.company,
                'person_name'   : c.person_name,
                'person_title'  : c.person_title,
                'person_email'  : c.person_email,
                'person_phone'  : c.person_phone,
                'error'         : None,
            })
        if err:
            history_rows.append({
                'run_ts'        : run_ts,
                'artist'        : row.artist,
                'rostr_handle'  : slug,
                'rostr_id'      : None,
                'role'          : None,
                'company'       : None,
                'person_name'   : None,
                'person_title'  : None,
                'person_email'  : None,
                'person_phone'  : None,
                'error'         : err,
            })

    if history_rows:
        df = spark.createDataFrame(history_rows)
        (df.write
           .mode('append')
           .saveAsTable(f'{UC_CATALOG}.{UC_SCHEMA}.team_enrichment_history'))
        print(f'Appended {len(history_rows)} rows to {UC_CATALOG}.{UC_SCHEMA}.team_enrichment_history')
    print(f'Raw payloads written under {base}/')
else:
    print('PERSIST_TO_UC=false — skipping UC writes.')

## Summary

In [ ]:
ok   = sum(1 for _, flat, _, _ in results if flat)
fail = sum(1 for _, _, _, err in results if err)
print('=' * 60)
print(f'  Considered      : {len(needs)} rows')
print(f'  Resolved        : {ok}')
print(f'  Failed lookups  : {fail}')
print(f'  Cells written   : {n_written}')
print(f'  Stamp           : {stamp}')
print('=' * 60)